In [ ]:
%reset -f

In [ ]:
import torch
# import torch.accelerator
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
import os
import pandas as pd
from typing import Callable

In [ ]:
try:
    device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
except AttributeError:
    device = "cpu"
device

In [ ]:
# Hyper parameters
batch_size = 64
learning_rate = 1e-3
epochs = 50

n = 20

In [ ]:
def list_from_str(str_list: str, fn_apply_to_items: Callable) -> list:
    if str_list in ["", "[]"]:
        return []

    elems = str_list.strip("[]\"").split(", ")
    return list(map(fn_apply_to_items, elems))

In [ ]:
class PIIStateDataset(Dataset):
    def __init__(self, data_file: str, rows_per_trial: int = 1, nrows: int | None = None, start_row: int = 0) -> None:
        self.rows_per_trial = rows_per_trial

        skiprows = range(1, start_row) if start_row > 0 else 0
        df = pd.read_csv(data_file, nrows=nrows, skiprows=skiprows)

        self.numUnstable = torch.tensor(df["numUnstable"], dtype=torch.float)
        # Normalize by dividing by n
        self.numUnstable = self.numUnstable.div(n)

        self.numNM1 = torch.tensor(df["numNM1"], dtype=torch.float)
        self.numNM2 = torch.tensor(df["numNM2"], dtype=torch.float)

        self.numEdges = torch.tensor(df["numEdges"], dtype=torch.float)
        self.numSingletons = torch.tensor(df["numSingletons"], dtype=torch.float)
        self.numChains = torch.tensor(df["numChains"], dtype=torch.float)
        self.numCycles = torch.tensor(df["numCycles"], dtype=torch.float)

        self.avgChainLen = torch.tensor(df["avgChainLen"], dtype=torch.float)
        self.avgCycleLen = torch.tensor(df["avgCycleLen"], dtype=torch.float)

        self.converges = torch.tensor(df["converges"], dtype=torch.long)
        self.convergesOneHot = nn.functional.one_hot(self.converges, 2).float()

        # TODO: Normalize data by dividing by n

        list_slices = [
            ("matchingAM", "matchingGM"),
            ("unstableAM", "unstableRAM"),
            ("nm1AM", "nm2GenRAM"),
            ("nm2AM", "nm2RAM"),
        ]

        self.all_mean_sequences = []
        for start_str, end_str in list_slices:
            start, end = df.columns.slice_locs(start_str, end_str)
            mean_columns = df.iloc[:, start:end]

            sequences = []
            for _, row in mean_columns.iterrows():
                list_of_mean_lists = [list_from_str(mean_list, float) for mean_list in row]
                sequences.append(torch.tensor(list_of_mean_lists).T)

            self.all_mean_sequences.append(sequences)

        self.singleton_features = torch.stack((
            self.numUnstable, self.numNM1, self.numNM2,
            self.numEdges, self.numSingletons, self.numChains, self.numCycles,
            self.avgChainLen, self.avgCycleLen
        ), dim=1)
        # Normalize by dividing by n
        # self.singleton_features = self.singleton_features.div(n)

        self.singleton_features = self.singleton_features.div(n)

    def __len__(self):
        return len(self.converges) // self.rows_per_trial

    def __getitem__(self, idx) -> tuple:
        seq_at_idx = [seq[idx] for seq in self.all_mean_sequences]
        X = (self.singleton_features[idx], seq_at_idx)

        return X, self.convergesOneHot[idx]

In [ ]:
separateFileData = True

if separateFileData:
    rows_per_trial = 1
    # dataset_str = "stateData_2000_10"
    # dataset_str = "stateData_2000_10_itr0"
    # dataset_str = "stateData_2000_10_itr1"
    # dataset_str = "stateData_2000_10_itr2"
    # dataset_str = "stateData_2000_10_itr_-1"

    # dataset_str = "balStateData_20000_10_itr0"
    # dataset_str = "balStateData_20000_10_itr1"
    # dataset_str = "balStateData_20000_10_itr2"
    # dataset_str = "balStateData_20000_10_itr_-1"

    # test_dataset_str = dataset_str + "_test"

    n = 20
    dataset_str = f"stateData_2000_{n}_ID0_itr0"
    test_dataset_str = f"stateData_2000_{n}_ID1_itr0"

    # rows_per_trial = 2
    # dataset_str = "balStateData_20000_10_itr_0+1"

    training_data = PIIStateDataset(f"data/{dataset_str}.csv", rows_per_trial)
    test_data = PIIStateDataset(f"data/{test_dataset_str}.csv", rows_per_trial)

    # train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    # test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
else:
    rows_per_trial = 1
    n = 20
    size = 20000
    dataset_str = f"stateData_{size}_{n}_itr0"

    train_ratio = 0.75

    training_data = PIIStateDataset(f"data/{dataset_str}.csv", rows_per_trial, int(train_ratio * size), 0)
    test_data = PIIStateDataset(f"data/{dataset_str}.csv", rows_per_trial, int((1 - train_ratio) * size), int(train_ratio * size))

    # train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
    # test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [ ]:
# training_data[0][0].shape

print(len(training_data), len(test_data))

In [ ]:
for i in range(2):
    print(i)
    print(training_data[i])

In [ ]:
for i in range(2):
    print(i)
    print(test_data[i])

In [ ]:
# # Check overlap between training and testing
# df_train = pd.read_csv(f"sign_{n}_{perm_total}.csv")
# df_test = pd.read_csv(f"sign_{n}_{perm_total}_test.csv")

# set(df_train["permutation"]).intersection(set(df_test["permutation"]))

In [ ]:
%reset_selective -f (model|loss_fn|optimizer)

In [ ]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self, input_size) -> None:
        super().__init__()
        # self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            # nn.Linear(16, 16),
            # nn.ReLU(),
            nn.Linear(16, 2),
            # nn.Softmax()
            # nn.ReLU()
        )

    def forward(self, x):
        # x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
class BasicLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1) -> None:
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers)
        self.ff = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.ff(out[-1, :])
        return out

In [ ]:
(singletons, mean_lists) ,_y = training_data[0]
feature_list = 14

lstms = []
for means in mean_lists:
    lstms.append(BasicLSTM(means.size(1), 64, 1).to(device))

ff_model = NeuralNetwork(singletons.size(0) + feature_list).to(device)

display(lstms)
display(ff_model)

In [ ]:
# loss_fn = nn.CrossEntropyLoss()
# loss_fn = nn.BCELoss()
loss_fn = nn.BCEWithLogitsLoss()

# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# optimizer = torch.optim.ASGD(model.parameters(), lr=learning_rate)
optimizer = torch.optim.Adam(ff_model.parameters(), lr=learning_rate)
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataset, model, loss_fn, optimizer):
    size = len(dataset)
    num_batches = len(dataset)
    train_loss, correct = 0, 0

    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for (singletons_X, mean_lists_X), y in dataset:
        singletons_X = singletons_X.to(device)
        y = y.to(device)

        feature_list = [singletons_X]
        for means, lstm_model in zip(mean_lists_X, lstms):
            means = means.to(device)
            feature_list.append(lstm_model(means))

        X = torch.cat(feature_list, dim=0)

        # Compute prediction and loss
        pred = model(X)

        loss = loss_fn(pred, y)
        train_loss += loss.item()

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        pred = pred.unsqueeze(dim=0)
        y = y.unsqueeze(dim=0)

        # display(pred)
        # display(y)

        correct += pred.argmax(1).eq(y.argmax(1)).sum().item()

    correct /= size
    train_loss /= num_batches

    return train_loss, correct


def test_loop(dataset, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataset)
    num_batches = len(dataset)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for (singletons_X, mean_lists_X), y in dataset:
            singletons_X = singletons_X.to(device)
            y = y.to(device)

            feature_list = [singletons_X]
            for means, lstm_model in zip(mean_lists_X, lstms):
                means = means.to(device)
                feature_list.append(lstm_model(means))

            X = torch.cat(feature_list, dim=0)

            pred: torch.Tensor = model(X)
            test_loss += loss_fn(pred, y).item()

            pred = pred.unsqueeze(dim=0)
            y = y.unsqueeze(dim=0)
            correct += pred.argmax(1).eq(y.argmax(1)).sum().item()


    test_loss /= num_batches
    correct /= size

    return test_loss, correct

In [ ]:
train_losses, train_accuracies = np.zeros(epochs, dtype=np.float32), np.empty(epochs, dtype=np.float32)
test_losses, test_accuracies = np.zeros(epochs, dtype=np.float32), np.empty(epochs, dtype=np.float32)

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss, train_accuracy = train_loop(training_data, ff_model, loss_fn, optimizer)
    test_loss, test_accuracy = test_loop(test_data, ff_model, loss_fn)

    train_losses[t] = train_loss
    train_accuracies[t] = train_accuracy
    test_losses[t] = test_loss
    test_accuracies[t] = test_accuracy

    print(f"Train Error: \n Accuracy: {(100*train_accuracy):>0.1f}%, Avg loss: {train_loss:>8f} \n")
    print(f"Test Error: \n Accuracy: {(100*test_accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")
print("Done!")

In [ ]:
def plot_data(
        x, ys: np.ndarray, labels: list[str] | None = None, title: str = "", ylabel: str = ""
    ) -> tuple:
    fig, ax = plt.subplots()

    if len(ys.shape) > 1:
        assert isinstance(labels, list)
        for y in ys:
            ax.plot(x, y)
    else:
        line = ax.plot(x, ys)
        ax.legend(handles=line)

    ax.set(xlabel="Epoch", ylabel=ylabel, title=title)
    ax.grid()

    if labels != None:
        ax.legend(labels)

    return fig, ax

In [ ]:
x = np.arange(0, epochs)

losses = np.vstack((train_losses, test_losses))
loss_fig, loss_ax = plot_data(x, losses, ["Training", "Testing"], f"Loss vs. Epoch ({dataset_str})", "Loss")
plt.savefig(f"plots/lstm_mean_{dataset_str}_loss")

accuracies = np.vstack((train_accuracies, test_accuracies)) * 100
acc_fig, acc_ax = plot_data(x, accuracies, ["Training", "Testing"], f"Accuracy vs. Epoch ({dataset_str})", "Accuracy (%)")
plt.savefig(f"plots/lstm_mean_{dataset_str}_acc")